# Folketing Open Data – Data Extraction and Exploration

I've made a notebook that explores data from Folketingets Åbne Data (ODA) and constructs a dataset of parliamentary roll-call votes for use in the project.

It follows the data-selection procedure published by Michele (https://www.michelecoscia.com/?page_id=2497) as closely as possible. Their code selects parliamentary cases (`Sag`) with `typeid == 3`, connects these to their procedural steps (`Sagstrin`), and then selects the relevant roll-call votes (`Afstemning`). The new addition is (`titel`) and (`titelkort`) of each case, as these will be used for NLP-based matching with the candidate-test questions. 

The main steps are:

1. Retrieve relevant parliamentary cases (`Sag`) and retain their textual descriptions.
2. Retrieve and connect procedural steps (`Sagstrin`) to the selected cases.
3. Retrieve roll calls (`Afstemning`) and apply the same selection criteria as the original analysis.

I believe the next steps are:

4. Retrieve individual votes (`Stemme`) for the selected roll calls.
5. Connect individual votes to politicians (`Aktør`).
6. Construct a final dataset linking politicians and their votes to the corresponding parliamentary case and its textual description.

Election-period filtering corresponding to FV11, FV15, FV19, and FV22 is a further step to be taken. 

In [2]:
import requests
import pandas as pd

BASE_URL = "https://oda.ft.dk/api"

In [3]:
def fetch_all(entity, params=None, page_size=100):
    params = params.copy() if params else {}

    rows = []
    skip = 0

    while True:
        query_params = {
            **params,
            "$top": page_size,
            "$skip": skip
        }

        response = requests.get(
            f"{BASE_URL}/{entity}",
            params=query_params
        )
        response.raise_for_status()

        batch = response.json()["value"]

        if not batch:
            break

        rows.extend(batch)

        if len(batch) < page_size:
            break

        skip += page_size

    return pd.DataFrame(rows)

1. Retrieve relevant parliamentary cases (`Sag`) and retain their textual descriptions.

In [4]:
df_sag = fetch_all(
    "Sag",
    params={
        "$filter": "typeid eq 3",
        "$select": "id,typeid,nummer,titel,titelkort"
    }
)

df_sag.head()

,id,typeid,nummer,titel,titelkort
0,66,3,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
1,68,3,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
2,69,3,L 107,Forslag til lov om Danmarks Innovationsfond.,Om Danmarks Innovationsfond.
3,70,3,L 109,Forslag til lov om ændring af lov om forskning...,Om konsekvensændringer som følge af lov om Dan...
4,71,3,L 108,Forslag til lov om ændring af lov om teknologi...,Om konsekvensændringer som følge af lov om Dan...


In [6]:
df_sag = df_sag.rename(columns={
    "id": "sagid",
    "typeid": "sagstypeid",
    "nummer": "sag_nummer",
    "titel": "sag_titel",
    "titelkort": "sag_titelkort"
})

In [7]:
print(df_sag.shape)
print(df_sag.columns)
df_sag.head()

(5261, 5)
Index(['sagid', 'sagstypeid', 'sag_nummer', 'sag_titel', 'sag_titelkort'], dtype='object')


,sagid,sagstypeid,sag_nummer,sag_titel,sag_titelkort
0,66,3,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
1,68,3,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
2,69,3,L 107,Forslag til lov om Danmarks Innovationsfond.,Om Danmarks Innovationsfond.
3,70,3,L 109,Forslag til lov om ændring af lov om forskning...,Om konsekvensændringer som følge af lov om Dan...
4,71,3,L 108,Forslag til lov om ændring af lov om teknologi...,Om konsekvensændringer som følge af lov om Dan...


2. Retrieve and connect procedural steps (`Sagstrin`) to the selected cases.

In [11]:
df_sagstrin = fetch_all(
    "Sagstrin",
    params={
        "$select": "id,dato,typeid,sagid"
    }
)

In [12]:
df_sagstrin = df_sagstrin.rename(columns={
    "id": "sagstrinid",
    "typeid": "sagstrintypeid"
})

In [13]:
df_sagstrin["dato"] = pd.to_datetime(
    df_sagstrin["dato"],
    errors="coerce"
)

In [14]:
df_procedures = df_sagstrin.merge(
    df_sag,
    on="sagid",
    how="inner"
)

In [15]:
df_procedures[
    [
        "sagstrinid",
        "dato",
        "sagstrintypeid",
        "sagid",
        "sag_nummer",
        "sag_titel",
        "sag_titelkort"
    ]
].head(20)

,sagstrinid,dato,sagstrintypeid,sagid,sag_nummer,sag_titel,sag_titelkort
0,135,2014-01-30 10:00:00,31,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
1,136,2014-01-30 00:00:00,32,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
2,137,2014-02-07 10:00:00,12,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
3,142,2014-02-27 00:00:00,14,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
4,144,2014-03-11 13:00:00,15,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
5,146,2014-03-13 10:00:00,17,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
6,147,2014-03-13 00:00:00,37,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
7,150,2014-01-15 13:00:00,31,68,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
8,151,2014-01-15 00:00:00,32,68,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
9,152,2014-01-23 10:00:00,12,68,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...


3. Retrieve roll calls (`Afstemning`) and apply the same selection criteria as the original analysis.

In [16]:
df_afstemning = fetch_all(
    "Afstemning",
    params={
        "$filter": "typeid eq 1",
        "$select": "id,sagstrinid,kommentar,typeid"
    }
)

In [17]:
df_afstemning = df_afstemning.rename(columns={
    "id": "afstemningid",
    "typeid": "afstemningstypeid"
})

In [18]:
df_afstemning = df_afstemning[
    df_afstemning["kommentar"].isna()
]

In [19]:
df_roll_calls = df_afstemning.merge(
    df_procedures,
    on="sagstrinid",
    how="inner"
)

In [20]:
df_roll_calls[
    [
        "afstemningid",
        "sagstrinid",
        "sagstrintypeid",
        "dato",
        "sagid",
        "sag_nummer",
        "sag_titel",
        "sag_titelkort"
    ]
].head(20)

,afstemningid,sagstrinid,sagstrintypeid,dato,sagid,sag_nummer,sag_titel,sag_titelkort
0,2,4849,17,2014-09-09 09:15:00,1449,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
1,1783,37246,17,2014-10-31 10:00:00,12715,L 4,Forslag til lov om ændring af lov om afgift af...,Om tilbagerulning af forsyningssikkerhedsafgif...
2,2267,38484,17,2014-12-02 13:00:00,13015,L 33,Forslag til lov om ændring af lov om produktio...,Om kompetencebevis.
3,2268,38341,17,2014-12-02 13:00:00,13004,L 19,Forslag til lov om dansk turisme.,Om dansk turisme.
4,2282,38172,17,2014-12-04 10:00:00,12991,L 13,Forslag til lov om ændring af lov om aktiv soc...,Om ændring af formue- og fradragsregler ved ef...
5,2283,38211,17,2014-12-04 10:00:00,12994,L 14,Forslag til lov om ændring af lov om ferie. (F...,Om færre betingelser for optjening af sygeferi...
6,2284,38185,17,2014-12-04 10:00:00,12992,L 15,Forslag til lov om ændring af lov om vikarers ...,Om overførsel af kompetencer til Arbejdsretten.
7,2285,38380,17,2014-12-04 10:00:00,13007,L 17,Forslag til lov om ophævelse af lov om nærings...,Om afskaffelse af næringsbrevsordningen.
8,2286,38353,17,2014-12-04 10:00:00,13005,L 18,Forslag til lov om ændring af konkurrenceloven...,Om ændring af konkurrenceloven m.v.
9,2287,54290,17,2014-12-04 10:00:00,20246,L 46,Forslag til lov om Danmarks Grønne Investering...,Om Danmarks Grønne Investeringsfond.


In [21]:
print("Antal roll calls:", len(df_roll_calls))
print("Unikke afstemninger:", df_roll_calls["afstemningid"].nunique())
print("Unikke sager:", df_roll_calls["sagid"].nunique())

print("\nSagstrin:")
print(df_roll_calls["sagstrintypeid"].value_counts(dropna=False))

print("\nDubletter pr. sag:")
print(
    df_roll_calls["sagid"]
    .value_counts()
    .head(10)
)

Antal roll calls: 2131
Unikke afstemninger: 2131
Unikke sager: 2131

Sagstrin:
sagstrintypeid
17    2131
Name: count, dtype: int64

Dubletter pr. sag:
sagid
105374    1
1449      1
12715     1
103326    1
103792    1
103407    1
103327    1
103791    1
103111    1
104028    1
Name: count, dtype: int64


## From parliamentary cases to final roll-call votes

The extraction connects three levels of the ODA data: parliamentary cases (`Sag`), procedural steps (`Sagstrin`), and roll-call votes (`Afstemning`).

### Parliamentary cases (`Sag`)

Following the selection used in the original analysis, only cases with `Sag.typeid == 3` are retained. In the retrieved data, these correspond to legislative proposals with case numbers beginning with `L`.

This results in 5,261 legislative cases. In addition to the identifiers used in the original analysis, the title (`titel`) and short title (`titelkort`) are retained. These fields provide textual descriptions of the subject of each proposal and will later be used for NLP-based comparison with candidate-test questions.

### Procedural steps (`Sagstrin`)

Each legislative case can contain multiple procedural steps. Joining `Sagstrin` to the selected cases therefore produces several observations for the same `sagid`.

For example, `L 105` appears multiple times with different `sagstrintypeid` values and dates. These observations represent different stages in the parliamentary treatment of the same proposal. One of these stages has `sagstrintypeid == 17`, corresponding to the third reading (`3. behandling`).

At this stage, the dataset therefore represents the parliamentary history of the selected legislative cases rather than one observation per case.

### Roll-call votes (`Afstemning`)

The procedural steps are subsequently joined to `Afstemning`. Following the original analysis, only `Afstemning.typeid == 1` is retained, while observations containing a value in `kommentar` are excluded.

After applying these criteria, the resulting dataset contains 2,131 roll calls.

All 2,131 selected roll calls are associated with `sagstrintypeid == 17`. The filtering procedure therefore results in roll calls taking place at the third reading of the legislative proposals.

Furthermore, there are:

- 2,131 roll calls (`afstemningid`)
- 2,131 unique parliamentary cases (`sagid`)
- no duplicated cases among the selected roll calls

Within this extracted dataset, there is therefore a one-to-one relationship between a selected legislative case and its final roll-call vote:

`Sag (legislative proposal) → Sagstrin (third reading) → Afstemning (roll-call vote)`

This is important for the subsequent analysis because each roll call can be associated unambiguously with the title and short title of a single legislative case. The next step is to connect these roll calls to the individual votes (`Stemme`) cast by parliamentary actors.